# 08 Generate GOBM Uniform-Domain Weights and DII Intermediates

Purpose: generate the full/uniform-domain GOBM feature-weight and DII intermediates used by the comparison plotting scripts.

Inputs:
- GOBM `.pkl` files under `gs://leap-persistent/vacquaviva/GOBMs/` or an equivalent local directory

Outputs:
- `Finalweights_sampled16k_<model>.txt`
- `Finalimbs_sampled16k_<model>.txt`

where `<model>` is `ETHZ`, `FESOM`, `IPSL`, `MRI`, and `NorESM`.

Estimated runtime: slow. Each model runs DADAPy backward greedy DII elimination on a 16k sample.

Notes:
- This notebook is a provenance workflow for generating full-domain GOBM weight and DII files.
- This notebook uses the full model domain after dropping duplicates and missing values.
- It does not apply the SOCAT mask; the SOCAT-mask model provenance is in `07_generate_gobm_socatmask_weights_dii_from_romy.ipynb`.
- To regenerate the archived 20k convergence inputs, change the sample size from 16,000 to 20,000 in the parameter cell and save outputs with `sampled20k` filenames.
- This optional regeneration step requires large upstream GOBM `.pkl` files and a compatible Python environment.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from dadapy.feature_weighting import FeatureWeighting
from sklearn.preprocessing import StandardScaler

# Use the GCS root by default. Set this to a local folder if the pkl files have been downloaded.
GOBM_ROOT = "gs://leap-persistent/vacquaviva/GOBMs"
OUTPUT_DIR = Path("../intermediates")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

N_SAMPLE = 16000
RANDOM_STATE = 13
N_EPOCHS = 80

# Keep this aligned with the comparison figures. Set to None to use all available years.
TIME_MIN = "2020-01-01"


## Model Files and Feature Definitions


In [ ]:
MODEL_FILES = {
    "ETHZ": "CESM_ETHZ/MLinput_CESM_ETHZ_mon_1x1_197001_202212.pkl",
    "FESOM": "FESOM2_REcoM/MLinput_FESOM2_REcoM_mon_1x1_197001_202212.pkl",
    "IPSL": "IPSL/MLinput_IPSL_mon_1x1_197001_202212.pkl",
    "MRI": "MRI_ESM2_2/MLinput_MRI_ESM2_2_mon_1x1_197001_202212.pkl",
    "NorESM": "NorESM/MLinput_NorESM_mon_1x1_197001_202212.pkl",
}

FEATURES = [
    "sst",
    "sst_anom",
    "sss",
    "sss_anom",
    "chl_log",
    "chl_log_anom",
    "mld_log",
    "xco2",
    "A",
    "B",
    "C",
    "T0",
    "T1",
]
TARGET = "delta_fco2_1D"
SOCAT_MASK = "socat_mask"


## Helper Functions


In [ ]:
def model_path(model):
    root = str(GOBM_ROOT).rstrip("/")
    return f"{root}/{MODEL_FILES[model]}"

def load_model_uniform_domain_scaled(model):
    print(f"Loading {model}")
    frame = pd.read_pickle(model_path(model))
    frame[TARGET] = frame["sfco2"] - frame["xco2"]
    frame = frame[FEATURES + [SOCAT_MASK, TARGET]]

    # No SOCAT-mask filter and no geographic latitude-band filter in this notebook.
    if TIME_MIN is not None:
        frame = frame[frame.index.get_level_values("time") >= TIME_MIN]

    scaled = StandardScaler().fit_transform(frame.loc[:, FEATURES + [TARGET]])
    scaled = pd.DataFrame(scaled, columns=FEATURES + [TARGET])
    scaled = scaled.drop_duplicates(subset=[TARGET]).dropna()
    print(model, "scaled rows:", scaled.shape[0])
    return scaled

def run_model(model, scaled):
    print(f"Starting {model}")
    sample = scaled.sample(n=N_SAMPLE, random_state=RANDOM_STATE)

    target = FeatureWeighting(sample[[TARGET]].to_numpy(), verbose=True)
    inputs = FeatureWeighting(sample[FEATURES].to_numpy(), verbose=True)

    final_imbs, final_weights = inputs.return_backward_greedy_dii_elimination(
        target_data=target,
        initial_weights=None,
        n_epochs=N_EPOCHS,
        learning_rate=None,
        decaying_lr="cos",
    )

    weights_path = OUTPUT_DIR / f"Finalweights_sampled16k_{model}.txt"
    imbs_path = OUTPUT_DIR / f"Finalimbs_sampled16k_{model}.txt"
    np.savetxt(weights_path, final_weights)
    np.savetxt(imbs_path, final_imbs)
    print(f"Saved {weights_path}")
    print(f"Saved {imbs_path}")


## Generate Outputs

Slow cell. It loops over all five models. If a run is interrupted, restrict `MODELS_TO_RUN` and rerun only missing models.


In [ ]:
MODELS_TO_RUN = list(MODEL_FILES)

for model in MODELS_TO_RUN:
    scaled_model = load_model_uniform_domain_scaled(model)
    run_model(model, scaled_model)


## Outputs Created

Quick check:


In [ ]:
expected_outputs = []
for model in MODEL_FILES:
    expected_outputs.extend([
        f"Finalweights_sampled16k_{model}.txt",
        f"Finalimbs_sampled16k_{model}.txt",
    ])

for filename in expected_outputs:
    path = OUTPUT_DIR / filename
    print(filename, "OK" if path.exists() else "MISSING")
